# PE posterior violins with CV-bias markers




## 1. Imports

In [1]:
import json
import re
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display

import lisabeta.utils.plotutils as plotutils

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 180)

# Avoid failures caused by an unavailable external LaTeX installation.
plt.rcParams["text.usetex"] = False
plt.rcParams["mathtext.fontset"] = "dejavusans"

/pbs/home/a/amahdi/lisabeta/src/lisabeta/waveforms/bbh/lalsim_wrap.py:10: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


## 2. Paths and settings

In [2]:
# ---------------------------------------------------------------------
# Main settings
# ---------------------------------------------------------------------

INJ_ID = 6

PE_BASE_DIR = Path("/pbs/home/a/amahdi/PE_for_my_work")
CASE_ID = f"M1e6_SEOBv5_id_{INJ_ID}"
CONFIG_DIR = PE_BASE_DIR / "config_files" / CASE_ID

CV_PATH_CANDIDATES = [
    Path(
        "/pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e6_extra/run_data/bias_syst_recovery/cv_bias_sigma_fti_all_params_all_inj_ids.csv"
    ),
]

OUTPUT_ROOT = CONFIG_DIR / "cv_posterior_violin_plots"
PER_RUN_DIR = OUTPUT_ROOT / "per_fti_run"
TABLE_DIR = OUTPUT_ROOT / "tables"

for directory in [OUTPUT_ROOT, PER_RUN_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# Raw-chain cleaning
BAD_LNL_CUT = -1e12

# Needed by helper functions copied from the corner notebook.
SIGMA_METHOD = "std"
MAX_CORNER_SAMPLES = 30000
CORNER_BINS = 25

# Plotting settings
MAX_VIOLIN_SAMPLES = 50000
RANDOM_SEED = 20260716
VIOLIN_QUANTILES = (0.05, 0.95)

# Set to (-50, 50) to reproduce the range used in the supervisor example.
# Set to None for automatic symmetric limits in each panel.
Y_LIMITS = (-50, 50)

SHOW_PE_MEDIAN = True
SHOW_PLUS_MINUS_ONE = True
SAVE_DPI = 250

# "truth": centre the PE samples on the injected parameters.
# This matches the convention in the existing inj_id comparison notebook.
#
# "reference_table": centre on values supplied in a CSV with columns
# active_fti_run, parameter, reference_value. This can be used for the
# aligned best-fit reference appearing in the supervisor's example.
CENTER_MODE = "truth"
REFERENCE_TABLE_PATH = None

PHYSICAL_PARAMETER_ORDER = [
    "Mchirp",
    "q",
    "chiPN",
    "chim",
    "Deltat",
    "dist",
    "inc",
    "phi",
    "lambda",
    "beta",
    "psi",
]

config_files = sorted(CONFIG_DIR.glob("*.json"))

if not config_files:
    raise FileNotFoundError(
        f"No JSON configuration files were found in {CONFIG_DIR}"
    )

print("Config directory:", CONFIG_DIR)
print("Number of config files:", len(config_files))
print("Output directory:", OUTPUT_ROOT)

Config directory: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6
Number of config files: 12
Output directory: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots


## 3. Locate and load processed/raw PE files

In [3]:
def infer_pe_base_dir_from_config_location(config_file):
    config_file = Path(config_file)
    return config_file.parent.parent.parent


def infer_run_data_root_from_config_location(config_file):
    return (
        infer_pe_base_dir_from_config_location(config_file)
        / 'run_data'
    )


def infer_run_dir_from_config_location(config_file):
    config_file = Path(config_file)
    return (
        infer_run_data_root_from_config_location(config_file)
        / config_file.parent.name
    )


def unique_labeled_paths(paths):
    unique = []
    seen = set()

    for label, path in paths:
        path = Path(path)
        key = str(path)

        if key not in seen:
            unique.append((label, path))
            seen.add(key)

    return unique


def candidate_output_paths(
    config_file,
    out_dir,
    out_name,
    suffix,
):
    config_file = Path(config_file)
    out_dir = Path(out_dir)

    run_data_root = (
        infer_run_data_root_from_config_location(config_file)
    )
    inferred_run_dir = (
        infer_run_dir_from_config_location(config_file)
    )
    case_id = config_file.parent.name

    candidates = [
        (
            'configured_string_concat',
            Path(str(out_dir) + f'{out_name}{suffix}'),
        ),
        (
            'inferred_string_concat',
            run_data_root / f'{case_id}{out_name}{suffix}',
        ),
        (
            'configured_path_join',
            out_dir / f'{out_name}{suffix}',
        ),
        (
            'inferred_path_join',
            inferred_run_dir / f'{out_name}{suffix}',
        ),
    ]

    return unique_labeled_paths(candidates)


def first_existing_path(candidates):
    for label, path in candidates:
        if path.exists():
            return label, path

    return None, None


def detect_active_fti_parameter(config):
    infer_params = list(
        config.get('prior_params', {})
        .get('infer_params', [])
    )

    configured_fti = (
        config.get('waveform_params', {})
        .get('tgr_params', {})
        .get('FTI', {})
        .get('fti_params', [])
    )

    active = [
        parameter
        for parameter in infer_params
        if parameter in configured_fti
        or parameter.startswith('fti_')
    ]

    if active:
        return active[-1]

    return None


def read_case_information(config_file):
    config_file = Path(config_file)

    with config_file.open('r') as stream:
        config = json.load(stream)

    run_params = config.get('run_params', {})
    infer_params = list(
        config.get('prior_params', {})
        .get('infer_params', [])
    )

    out_dir = Path(run_params.get('out_dir', ''))
    out_name = run_params.get(
        'out_name',
        config_file.stem,
    )

    processed_candidates = candidate_output_paths(
        config_file,
        out_dir,
        out_name,
        '.h5',
    )
    raw_candidates = candidate_output_paths(
        config_file,
        out_dir,
        out_name,
        '_raw.h5',
    )

    processed_kind, processed_file = (
        first_existing_path(processed_candidates)
    )
    raw_kind, raw_file = (
        first_existing_path(raw_candidates)
    )

    return {
        'config_file': config_file,
        'config': config,
        'run_params': run_params,
        'infer_params': infer_params,
        'active_fti': detect_active_fti_parameter(config),
        'out_dir': out_dir,
        'out_name': out_name,
        'processed_candidates': processed_candidates,
        'raw_candidates': raw_candidates,
        'processed_kind': processed_kind,
        'processed_file': processed_file,
        'raw_kind': raw_kind,
        'raw_file': raw_file,
    }


def load_raw_file(raw_file, infer_params):
    posterior = {}

    with h5py.File(raw_file, 'r') as h5:
        for parameter in infer_params:
            if parameter in h5:
                posterior[parameter] = h5[parameter][:]
            else:
                print(
                    f'[warning] {parameter} is absent '
                    f'from {raw_file}'
                )

        if 'lnlike' in h5:
            posterior['lnL'] = h5['lnlike'][:]
        elif 'lnL' in h5:
            posterior['lnL'] = h5['lnL'][:]
        else:
            raise KeyError(
                f"Neither 'lnlike' nor 'lnL' exists in {raw_file}"
            )

    return posterior


def load_case(config_file):
    case = read_case_information(config_file)

    case['processed_posterior'] = None
    case['raw_posterior'] = None

    if case['processed_file'] is not None:
        try:
            case['processed_posterior'] = (
                plotutils.load_params_posterior_lisa_smbh(
                    str(case['config_file']),
                    str(case['processed_file']),
                )
            )
        except Exception as error:
            print(
                '[warning] processed posterior could not be loaded '
                f"for {case['active_fti']}: {error}"
            )

    if case['raw_file'] is not None:
        try:
            case['raw_posterior'] = load_raw_file(
                case['raw_file'],
                case['infer_params'],
            )
        except Exception as error:
            print(
                '[warning] raw posterior could not be loaded '
                f"for {case['active_fti']}: {error}"
            )

    return case


In [4]:
cases = [load_case(path) for path in config_files]

for case in cases:
    print('=' * 100)
    print('Active FTI parameter:', case['active_fti'])
    print('Config:', case['config_file'])
    print('Processed posterior:', case['processed_file'])
    print('Raw posterior:', case['raw_file'])
    print('Inferred parameters:', case['infer_params'])

    if (
        case['processed_posterior'] is None
        and case['raw_posterior'] is None
    ):
        print('[warning] no usable posterior was loaded')


Active FTI parameter: fti_dchi0v
Config: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/M1e6_SEOBv5_id_6_fti_dchi0v_IMR.json
Processed posterior: /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_6M1e6_SEOBv5_id_6_fti_dchi0v_IMR.h5
Raw posterior: /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_6M1e6_SEOBv5_id_6_fti_dchi0v_IMR_raw.h5
Inferred parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi0v']
Active FTI parameter: fti_dchi1v
Config: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/M1e6_SEOBv5_id_6_fti_dchi1v_IMR.json
Processed posterior: /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_6M1e6_SEOBv5_id_6_fti_dchi1v_IMR.h5
Raw posterior: /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_6M1e6_SEOBv5_id_6_fti_dchi1v_IMR_raw.h5
Inferred parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi1v']
Active FTI

## 4. Posterior extraction

In [5]:
def processed_has_parameter(case, parameter):
    processed = case.get('processed_posterior')

    if processed is None:
        return False

    try:
        return parameter in processed['post']
    except Exception:
        return False


def raw_has_parameter(case, parameter):
    raw = case.get('raw_posterior')

    try:
        return raw is not None and parameter in raw
    except Exception:
        return False


def choose_corner_source(case, parameters):
    active_fti = case['active_fti']

    processed_available = [
        parameter
        for parameter in parameters
        if processed_has_parameter(case, parameter)
    ]
    raw_available = [
        parameter
        for parameter in parameters
        if raw_has_parameter(case, parameter)
    ]

    if (
        active_fti in processed_available
        and len(processed_available) == len(parameters)
    ):
        return 'processed', processed_available

    if active_fti in raw_available:
        missing = [
            parameter
            for parameter in parameters
            if parameter not in raw_available
        ]

        if missing:
            print(
                f'[warning] raw posterior for {active_fti} '
                f'is missing {missing}'
            )

        return 'raw', raw_available

    if active_fti in processed_available:
        missing = [
            parameter
            for parameter in parameters
            if parameter not in processed_available
        ]

        if missing:
            print(
                f'[warning] processed posterior for {active_fti} '
                f'is missing {missing}'
            )

        return 'processed', processed_available

    print(
        f'[error] active FTI parameter {active_fti} is absent '
        'from both available posterior sources'
    )
    return None, []


def decide_raw_burnin(case):
    raw = case.get('raw_posterior')

    if raw is None or 'lnL' not in raw:
        return 0, 'no raw likelihood'

    number_of_steps = int(
        np.asarray(raw['lnL']).shape[-1]
    )

    run_params = case.get('run_params', {})

    try:
        n_iter = (
            int(run_params['n_iter'])
            if run_params.get('n_iter') is not None
            else None
        )
    except Exception:
        n_iter = None

    try:
        burn_in = int(
            run_params.get('burn_in', 0) or 0
        )
    except Exception:
        burn_in = 0

    if burn_in <= 0:
        return 0, 'burn_in is zero or missing'

    if burn_in >= number_of_steps:
        return 0, (
            f'burn_in={burn_in} is not smaller than '
            f'raw nsteps={number_of_steps}'
        )

    if (
        n_iter is not None
        and number_of_steps == n_iter
    ):
        return burn_in, (
            f'raw nsteps={number_of_steps} equals '
            f'configured n_iter={n_iter}'
        )

    if (
        n_iter is not None
        and number_of_steps == max(n_iter - burn_in, 0)
    ):
        return 0, (
            'raw posterior already appears to have burn-in '
            'removed'
        )

    return 0, (
        'raw-chain length is ambiguous; no additional '
        'burn-in was removed'
    )


def slice_last_axis(array, start=0):
    array = np.asarray(array)

    index = [slice(None)] * array.ndim
    index[-1] = slice(start, None)

    return array[tuple(index)]


def align_array_to_likelihood(array, likelihood):
    array = np.asarray(array)
    likelihood = np.asarray(likelihood)

    if array.shape == likelihood.shape:
        return array

    if array.size == likelihood.size:
        return array.reshape(likelihood.shape)

    return None


def extract_processed_matrix(case, parameters):
    post = case['processed_posterior']['post']

    arrays = [
        np.asarray(post[parameter]).reshape(-1)
        for parameter in parameters
    ]

    lengths = [array.size for array in arrays]

    if len(set(lengths)) != 1:
        raise ValueError(
            'Processed posterior parameter lengths differ: '
            f'{dict(zip(parameters, lengths))}'
        )

    finite_mask = np.ones(lengths[0], dtype=bool)

    for array in arrays:
        finite_mask &= np.isfinite(array)

    samples = np.column_stack(
        [array[finite_mask] for array in arrays]
    )

    return samples, {
        'source': 'processed',
        'burnin_used': 0,
        'burnin_reason': 'processed posterior',
    }


def extract_raw_matrix(case, parameters):
    raw = case['raw_posterior']
    likelihood_full = np.asarray(raw['lnL'])

    burnin, burnin_reason = decide_raw_burnin(case)

    likelihood = slice_last_axis(
        likelihood_full,
        start=burnin,
    )

    mask = (
        np.isfinite(likelihood)
        & (likelihood > BAD_LNL_CUT)
    )

    arrays = []

    for parameter in parameters:
        aligned = align_array_to_likelihood(
            raw[parameter],
            likelihood_full,
        )

        if aligned is None:
            raise ValueError(
                f"{parameter} with shape "
                f"{np.asarray(raw[parameter]).shape} cannot "
                f"be aligned to likelihood shape "
                f"{likelihood_full.shape}"
            )

        sliced = slice_last_axis(
            aligned,
            start=burnin,
        )

        mask &= np.isfinite(sliced)
        arrays.append(sliced)

    samples = np.column_stack(
        [array[mask] for array in arrays]
    )

    return samples, {
        'source': 'raw',
        'burnin_used': burnin,
        'burnin_reason': burnin_reason,
    }


def extract_corner_matrix(case, source, parameters):
    if source == 'processed':
        return extract_processed_matrix(
            case,
            parameters,
        )

    if source == 'raw':
        return extract_raw_matrix(
            case,
            parameters,
        )

    raise ValueError(f'Unknown source: {source}')


## 5. Injected/reference values

In [6]:
def get_true_value(case, parameter):
    processed = case.get('processed_posterior')

    if processed is not None:
        injected = processed.get('injparams_Lframe', {})

        try:
            if parameter in injected:
                value = float(
                    np.asarray(injected[parameter]).squeeze()
                )

                if np.isfinite(value):
                    return value
        except Exception:
            pass

    config = case['config']
    source_params = config.get('source_params', {})

    try:
        if parameter in source_params:
            value = float(
                np.asarray(source_params[parameter]).squeeze()
            )

            if np.isfinite(value):
                return value
    except Exception:
        pass

    fti_block = (
        config.get('waveform_params', {})
        .get('tgr_params', {})
        .get('FTI', {})
    )

    if (
        parameter == 'Mchirp'
        and fti_block.get('Mchirp_inj') is not None
    ):
        return float(fti_block['Mchirp_inj'])

    if (
        parameter == 'fti_dxi0v'
        and fti_block.get('dxi0v_inj') is not None
    ):
        return float(fti_block['dxi0v_inj'])

    if parameter == case.get('active_fti'):
        return 0.0

    return None


def calculate_sigma(samples):
    samples = np.asarray(samples)

    if samples.size < 2:
        return np.nan

    if SIGMA_METHOD == 'std':
        return float(
            np.std(samples, ddof=1)
        )

    if SIGMA_METHOD == 'central68':
        percentile_16, percentile_84 = np.percentile(
            samples,
            [16.0, 84.0],
        )
        return float(
            0.5
            * (percentile_84 - percentile_16)
        )

    raise ValueError(
        "SIGMA_METHOD must be 'std' or 'central68'"
    )


def make_normalized_matrix(
    case,
    samples,
    parameters,
):
    normalized_columns = []
    retained_parameters = []
    metadata_rows = []

    for column_index, parameter in enumerate(parameters):
        parameter_samples = samples[:, column_index]

        true_value = get_true_value(
            case,
            parameter,
        )

        if (
            true_value is None
            or not np.isfinite(true_value)
        ):
            print(
                f'[warning] skipping {parameter}: '
                'true value was not found'
            )
            continue

        sigma_pe = calculate_sigma(
            parameter_samples
        )

        if (
            not np.isfinite(sigma_pe)
            or sigma_pe <= 0
        ):
            print(
                f'[warning] skipping {parameter}: '
                f'invalid sigma_PE={sigma_pe}'
            )
            continue

        normalized = (
            parameter_samples - true_value
        ) / sigma_pe

        normalized_columns.append(normalized)
        retained_parameters.append(parameter)

        percentile_16, median, percentile_84 = np.percentile(
            parameter_samples,
            [16.0, 50.0, 84.0],
        )

        metadata_rows.append({
            'active_fti_run': case['active_fti'],
            'parameter': parameter,
            'true_value': true_value,
            'sigma_method': SIGMA_METHOD,
            'sigma_PE': sigma_pe,
            'posterior_mean': float(
                np.mean(parameter_samples)
            ),
            'posterior_median': float(median),
            'median_minus_true': float(
                median - true_value
            ),
            'posterior_percentile_16': float(
                percentile_16
            ),
            'posterior_percentile_84': float(
                percentile_84
            ),
            'mean_bias_over_sigma_PE': float(
                (
                    np.mean(parameter_samples)
                    - true_value
                )
                / sigma_pe
            ),
            'median_bias_over_sigma_PE': float(
                (median - true_value)
                / sigma_pe
            ),
        })

    if len(normalized_columns) < 2:
        raise ValueError(
            'Fewer than two parameters have both a true value '
            'and a valid posterior sigma.'
        )

    normalized_matrix = np.column_stack(
        normalized_columns
    )

    return (
        normalized_matrix,
        retained_parameters,
        metadata_rows,
    )


def normalized_axis_label(parameter):
    # Use ASCII source plus Matplotlib mathtext commands, not Unicode Δ or σ.
    safe_parameter = str(parameter).replace('_', r'\_')
    return (
        rf'$\Delta\,\mathrm{{{safe_parameter}}}'
        rf'/\sigma_{{\mathrm{{PE}}}}$'
    )


def downsample_rows(matrix):
    if (
        MAX_CORNER_SAMPLES is None
        or len(matrix) <= MAX_CORNER_SAMPLES
    ):
        return matrix

    random_generator = np.random.default_rng(
        RANDOM_SEED
    )

    indices = random_generator.choice(
        len(matrix),
        size=MAX_CORNER_SAMPLES,
        replace=False,
    )

    return matrix[indices]


In [7]:
reference_table = None

if CENTER_MODE == "reference_table":
    if REFERENCE_TABLE_PATH is None:
        raise ValueError(
            "CENTER_MODE='reference_table' requires "
            "REFERENCE_TABLE_PATH."
        )

    reference_table = pd.read_csv(REFERENCE_TABLE_PATH)

    required = {
        "active_fti_run",
        "parameter",
        "reference_value",
    }
    missing = required.difference(reference_table.columns)

    if missing:
        raise ValueError(
            f"Reference table is missing columns: {sorted(missing)}"
        )

    if reference_table.duplicated(
        ["active_fti_run", "parameter"]
    ).any():
        raise ValueError(
            "Duplicate active_fti_run/parameter entries exist "
            "in the reference table."
        )


def get_reference_value(case, parameter):
    if CENTER_MODE == "truth":
        value = get_true_value(case, parameter)

    elif CENTER_MODE == "reference_table":
        rows = reference_table.loc[
            (reference_table["active_fti_run"] == case["active_fti"])
            & (reference_table["parameter"] == parameter)
        ]

        if len(rows) != 1:
            raise KeyError(
                f"Expected one reference row for "
                f"{case['active_fti']} / {parameter}; found {len(rows)}."
            )

        value = float(rows.iloc[0]["reference_value"])

    else:
        raise ValueError(
            "CENTER_MODE must be 'truth' or 'reference_table'."
        )

    if value is None or not np.isfinite(value):
        raise ValueError(
            f"No finite reference value for "
            f"{case['active_fti']} / {parameter}."
        )

    return float(value)

## 6. Load the CV/Fisher table

In [8]:
def resolve_existing_path(candidates, label):
    for candidate in candidates:
        candidate = Path(candidate)

        if candidate.exists():
            print(f"{label}: {candidate.resolve()}")
            return candidate

    checked = "\n".join(f"  - {Path(path)}" for path in candidates)

    raise FileNotFoundError(
        f"Could not find {label}. Paths checked:\n{checked}"
    )


CV_PATH = resolve_existing_path(
    CV_PATH_CANDIDATES,
    "CV-bias table",
)

cv_all = pd.read_csv(CV_PATH)

required_cv_columns = {
    "inj_id",
    "fti_case",
    "parameter",
    "Delta_theta_over_sigma",
    "Delta_theta",
    "sigma",
}

missing = required_cv_columns.difference(cv_all.columns)

if missing:
    raise ValueError(
        f"CV table is missing columns: {sorted(missing)}"
    )

cv = cv_all.loc[cv_all["inj_id"] == INJ_ID].copy()

if cv.empty:
    raise ValueError(
        f"No CV rows were found for inj_id={INJ_ID}."
    )

if cv.duplicated(["fti_case", "parameter"]).any():
    duplicates = cv.loc[
        cv.duplicated(
            ["fti_case", "parameter"],
            keep=False,
        )
    ].sort_values(["fti_case", "parameter"])

    display(duplicates)
    raise ValueError(
        "Duplicate fti_case/parameter rows were found."
    )

cv_lookup = cv.set_index(
    ["fti_case", "parameter"]
).sort_index()

print("CV rows retained:", len(cv))
print("CV FTI cases:", cv["fti_case"].nunique())
print(sorted(cv["fti_case"].unique()))

CV-bias table: /pbs/home/a/amahdi/pySEOBNR_deviation/Injection set/SEOBNRv5HMROM_times_pSEOB_ratio_phase_with_denom_condition_domega_dtau_scan_M1e6_extra/run_data/bias_syst_recovery/cv_bias_sigma_fti_all_params_all_inj_ids.csv
CV rows retained: 144
CV FTI cases: 12
['fti_dchi0v', 'fti_dchi1v', 'fti_dchi2v', 'fti_dchi3v', 'fti_dchi4v', 'fti_dchi5lv', 'fti_dchi6lv', 'fti_dchi6v', 'fti_dchi7v', 'fti_dchiMinus2v', 'fti_dkappa_S', 'fti_dxi0v']


## 7. Parameter labels and ordering

In [9]:
PHYSICAL_LABELS = {
    "Mchirp": r"$\mathcal{M}_c$",
    "q": r"$q$",
    "chiPN": r"$\chi_{\rm PN}$",
    "chim": r"$\chi_{-}$",
    "Deltat": r"$\Delta t$",
    "dist": r"$d_L$",
    "inc": r"$\iota$",
    "phi": r"$\phi$",
    "lambda": r"$\lambda$",
    "beta": r"$\beta$",
    "psi": r"$\psi$",
}


def parameter_label(parameter):
    if parameter in PHYSICAL_LABELS:
        return PHYSICAL_LABELS[parameter]

    name = str(parameter)

    if name == "fti_dxi0v":
        return r"$\delta\hat{\xi}_{0}$"

    if "kappa" in name.lower():
        return r"$\delta\kappa_s$"

    match = re.fullmatch(
        r"fti_dchi(?P<order>Minus\d+|\d+)(?P<log>l?)v",
        name,
    )

    if match:
        order = match.group("order")

        if order.startswith("Minus"):
            order = "-" + order.removeprefix("Minus")

        log_suffix = r"\ell" if match.group("log") else ""

        return (
            rf"$\delta\hat{{\varphi}}_"
            rf"{{{order}{log_suffix}}}$"
        )

    safe = name.replace("_", r"\_")
    return rf"$\mathrm{{{safe}}}$"


def ordered_parameters_for_case(case):
    inferred = list(dict.fromkeys(case["infer_params"]))
    active_fti = case["active_fti"]

    ordered = [
        parameter
        for parameter in PHYSICAL_PARAMETER_ORDER
        if parameter in inferred
    ]

    remaining = [
        parameter
        for parameter in inferred
        if parameter not in ordered
        and parameter != active_fti
        and not str(parameter).startswith("fti_")
    ]

    ordered.extend(remaining)

    if active_fti is not None and active_fti in inferred:
        ordered.append(active_fti)

    return ordered


def downsample_values(values, maximum, seed):
    values = np.asarray(values)

    if maximum is None or len(values) <= maximum:
        return values

    rng = np.random.default_rng(seed)
    indices = rng.choice(
        len(values),
        size=maximum,
        replace=False,
    )

    return values[indices]

## 8. Build normalized posterior arrays

In [10]:
plot_cases = []
summary_rows = []
skipped_rows = []

for case_index, case in enumerate(cases):
    active_fti = case["active_fti"]

    if active_fti is None:
        skipped_rows.append({
            "config_file": str(case["config_file"]),
            "reason": "No active FTI parameter detected",
        })
        continue

    parameters = ordered_parameters_for_case(case)

    if not parameters:
        skipped_rows.append({
            "config_file": str(case["config_file"]),
            "active_fti_run": active_fti,
            "reason": "No parameters selected",
        })
        continue

    try:
        source, available_parameters = choose_corner_source(
            case,
            parameters,
        )

        if source is None:
            raise ValueError(
                "Neither processed nor raw posterior contains "
                "the active FTI parameter."
            )

        # Keep only parameters available in the selected common source.
        selected_parameters = [
            parameter
            for parameter in parameters
            if parameter in available_parameters
        ]

        sample_matrix, source_metadata = extract_corner_matrix(
            case,
            source,
            selected_parameters,
        )

    except Exception as error:
        skipped_rows.append({
            "config_file": str(case["config_file"]),
            "active_fti_run": active_fti,
            "reason": str(error),
        })
        continue

    normalized_full = []
    normalized_plot = []
    retained_parameters = []
    cv_points = []
    pe_medians = []

    for parameter_index, parameter in enumerate(
        selected_parameters
    ):
        key = (active_fti, parameter)

        if key not in cv_lookup.index:
            print(
                f"[warning] no CV row for "
                f"{active_fti} / {parameter}; skipped."
            )
            continue

        cv_row = cv_lookup.loc[key]

        sigma_cv = float(cv_row["sigma"])
        delta_theta_cv = float(cv_row["Delta_theta"])

        if not np.isfinite(sigma_cv) or sigma_cv <= 0:
            print(
                f"[warning] invalid sigma_CV={sigma_cv} for "
                f"{active_fti} / {parameter}; skipped."
            )
            continue

        reference_value = get_reference_value(
            case,
            parameter,
        )

        samples = np.asarray(
            sample_matrix[:, parameter_index],
            dtype=float,
        )
        samples = samples[np.isfinite(samples)]

        if samples.size < 2:
            print(
                f"[warning] fewer than two samples for "
                f"{active_fti} / {parameter}; skipped."
            )
            continue

        normalized = (
            samples - reference_value
        ) / sigma_cv

        cv_point = delta_theta_cv / sigma_cv
        stored_cv_point = float(
            cv_row["Delta_theta_over_sigma"]
        )

        if not np.isclose(
            cv_point,
            stored_cv_point,
            rtol=1e-7,
            atol=1e-10,
        ):
            print(
                f"[warning] recomputed and stored CV ratios differ "
                f"for {active_fti} / {parameter}."
            )

        plot_values = downsample_values(
            normalized,
            MAX_VIOLIN_SAMPLES,
            RANDOM_SEED
            + 1000 * case_index
            + parameter_index,
        )

        q05, q16, median, q84, q95 = np.quantile(
            normalized,
            [0.05, 0.16, 0.50, 0.84, 0.95],
        )

        retained_parameters.append(parameter)
        normalized_full.append(normalized)
        normalized_plot.append(plot_values)
        cv_points.append(cv_point)
        pe_medians.append(median)

        summary_rows.append({
            "inj_id": INJ_ID,
            "active_fti_run": active_fti,
            "parameter": parameter,
            "posterior_source": source,
            "reference_mode": CENTER_MODE,
            "reference_value": reference_value,
            "sigma_CV": sigma_cv,
            "CV_delta_theta": delta_theta_cv,
            "CV_bias_over_sigma": cv_point,
            "number_of_posterior_samples": int(len(normalized)),
            "PE_mean_over_sigma_CV": float(np.mean(normalized)),
            "PE_median_over_sigma_CV": float(median),
            "PE_q05_over_sigma_CV": float(q05),
            "PE_q16_over_sigma_CV": float(q16),
            "PE_q84_over_sigma_CV": float(q84),
            "PE_q95_over_sigma_CV": float(q95),
            "PE_median_minus_CV": float(median - cv_point),
            "raw_burnin_used": source_metadata["burnin_used"],
            "raw_burnin_reason": source_metadata["burnin_reason"],
        })

    if retained_parameters:
        plot_cases.append({
            "case": case,
            "active_fti": active_fti,
            "parameters": retained_parameters,
            "labels": [
                parameter_label(parameter)
                for parameter in retained_parameters
            ],
            "normalized_full": normalized_full,
            "normalized_plot": normalized_plot,
            "cv_points": np.asarray(cv_points),
            "pe_medians": np.asarray(pe_medians),
            "posterior_source": source,
        })


summary_table = pd.DataFrame(summary_rows)

summary_path = (
    TABLE_DIR
    / f"posterior_violin_summary_inj_id_{INJ_ID}.csv"
)
summary_table.to_csv(summary_path, index=False)

print("Usable FTI runs:", len(plot_cases))
print("Summary rows:", len(summary_table))
print("Saved:", summary_path)

if skipped_rows:
    skipped_table = pd.DataFrame(skipped_rows)
    skipped_path = (
        TABLE_DIR
        / f"skipped_cases_inj_id_{INJ_ID}.csv"
    )
    skipped_table.to_csv(skipped_path, index=False)
    display(skipped_table)
    print("Saved:", skipped_path)

display(summary_table.head(30))

[error] active FTI parameter fti_dchi5lv is absent from both available posterior sources
Usable FTI runs: 11
Summary rows: 132
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/tables/posterior_violin_summary_inj_id_6.csv


,config_file,active_fti_run,reason
0,/pbs/home/a/amahdi/PE_for_my_work/config_files...,fti_dchi5lv,Neither processed nor raw posterior contains t...


Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/tables/skipped_cases_inj_id_6.csv


,inj_id,active_fti_run,parameter,posterior_source,reference_mode,reference_value,sigma_CV,CV_delta_theta,CV_bias_over_sigma,number_of_posterior_samples,PE_mean_over_sigma_CV,PE_median_over_sigma_CV,PE_q05_over_sigma_CV,PE_q16_over_sigma_CV,PE_q84_over_sigma_CV,PE_q95_over_sigma_CV,PE_median_minus_CV,raw_burnin_used,raw_burnin_reason
0,6,fti_dchi0v,Mchirp,processed,truth,333021.282961,89.944888,-132.721743,-1.475590,40000,-1.532898,-1.537544,-3.193427,-2.539622,-0.525250,0.136590,-0.061955,0,processed posterior
1,6,fti_dchi0v,q,processed,truth,4.000000,0.004664,-0.142289,-30.507287,40000,-29.087212,-29.086882,-30.651584,-30.024748,-28.150399,-27.543106,1.420405,0,processed posterior
2,6,fti_dchi0v,chiPN,processed,truth,0.446195,0.000302,-0.007225,-23.924936,40000,-23.718402,-23.701459,-25.395541,-24.728643,-22.708170,-22.094368,0.223477,0,processed posterior
3,6,fti_dchi0v,chim,processed,truth,0.300000,0.002627,-0.020482,-7.797017,40000,-7.133066,-7.129291,-8.719294,-8.104808,-6.169906,-5.545516,0.667726,0,processed posterior
4,6,fti_dchi0v,Deltat,processed,truth,0.000000,0.095337,2.013462,21.119421,40000,13.596110,13.592157,11.873494,12.558889,14.643416,15.331864,-7.527264,0,processed posterior
5,6,fti_dchi0v,dist,processed,truth,6791.810624,27.896084,-250.124597,-8.966298,40000,-9.415440,-9.413577,-11.276784,-10.535087,-8.292927,-7.582426,-0.447279,0,processed posterior
6,6,fti_dchi0v,inc,processed,truth,1.047198,0.002839,0.027522,9.694552,40000,9.349457,9.355012,7.674383,8.329339,10.367542,11.022590,-0.339540,0,processed posterior
7,6,fti_dchi0v,phi,processed,truth,0.700000,0.015780,-0.454792,-28.820241,40000,-27.317071,-27.304353,-28.897527,-28.283575,-26.362586,-25.751720,1.515888,0,processed posterior
8,6,fti_dchi0v,lambda,processed,truth,1.000000,0.003374,-0.030223,-8.958059,40000,-8.835216,-8.826607,-10.585206,-9.914483,-7.765598,-7.094471,0.131452,0,processed posterior
9,6,fti_dchi0v,beta,processed,truth,0.523599,0.001704,-0.003114,-1.827916,40000,-3.072218,-3.072195,-4.855590,-4.139832,-2.011248,-1.273815,-1.244279,0,processed posterior


## 9. Plotting functions

In [11]:
def automatic_y_limits(sample_lists, cv_points):
    values = []

    for samples in sample_lists:
        samples = np.asarray(samples)

        if samples.size:
            values.extend(
                np.quantile(
                    samples,
                    [0.005, 0.995],
                ).tolist()
            )

    finite_cv = np.asarray(cv_points, dtype=float)
    values.extend(
        finite_cv[np.isfinite(finite_cv)].tolist()
    )

    if not values:
        return (-1.0, 1.0)

    maximum = max(
        abs(min(values)),
        abs(max(values)),
    )
    maximum = max(1.0, 1.1 * maximum)

    return (-maximum, maximum)


def make_violin_figure(
    normalized_plot,
    normalized_full,
    cv_points,
    pe_medians,
    labels,
    title,
):
    positions = np.arange(1, len(labels) + 1)
    quantiles = [
        list(VIOLIN_QUANTILES)
        for _ in labels
    ]

    fig_width = max(12, 1.05 * len(labels))
    fig, ax = plt.subplots(
        figsize=(fig_width, 6.5)
    )

    violin = ax.violinplot(
        normalized_plot,
        positions=positions,
        showextrema=False,
        quantiles=quantiles,
    )

    posterior_color = plotutils.plotpalette[0]
    cv_color = plotutils.plotpalette[1]

    for body in violin["bodies"]:
        body.set_facecolor(posterior_color)
        body.set_edgecolor(posterior_color)
        body.set_alpha(0.65)

    if "cquantiles" in violin:
        violin["cquantiles"].set_color(
            posterior_color
        )
        violin["cquantiles"].set_linewidth(1.1)

    ax.scatter(
        positions,
        cv_points,
        color=cv_color,
        s=58,
        zorder=5,
    )

    if SHOW_PE_MEDIAN:
        ax.scatter(
            positions,
            pe_medians,
            marker="_",
            color="black",
            s=170,
            linewidths=1.6,
            zorder=6,
        )

    for position in positions:
        ax.axvline(
            position,
            color="black",
            linewidth=0.45,
            alpha=0.65,
            zorder=0,
        )

    ax.axhline(
        0.0,
        color="black",
        linewidth=1.0,
    )

    if SHOW_PLUS_MINUS_ONE:
        for level in (-1.0, 1.0):
            ax.axhline(
                level,
                color="black",
                linestyle="--",
                linewidth=0.7,
                alpha=0.65,
            )

    limits = (
        Y_LIMITS
        if Y_LIMITS is not None
        else automatic_y_limits(
            normalized_full,
            cv_points,
        )
    )
    ax.set_ylim(limits)

    ax.set_xlim(0.5, len(labels) + 0.5)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=11)

    ax.set_ylabel(
        r"$(\theta-\theta_{\rm ref})/"
        r"\sigma_{\rm CV}$"
    )
    ax.set_title(title)

    handles = [
        plt.Line2D(
            [0],
            [0],
            color=posterior_color,
            linewidth=8,
            alpha=0.65,
            label="PE posterior",
        ),
        plt.Line2D(
            [],
            [],
            marker="o",
            markersize=7,
            color=cv_color,
            linestyle="None",
            label="CV bias",
        ),
    ]

    if SHOW_PE_MEDIAN:
        handles.append(
            plt.Line2D(
                [],
                [],
                marker="_",
                markersize=13,
                markeredgewidth=1.6,
                color="black",
                linestyle="None",
                label="PE median",
            )
        )

    ax.legend(
        handles=handles,
        loc="upper center",
        ncol=len(handles),
    )

    ax.grid(False)
    fig.tight_layout()

    return fig, ax


def plot_one_fti_run(
    plot_case,
    save=True,
    show=True,
):
    active_fti = plot_case["active_fti"]

    title = (
        f"{parameter_label(active_fti)} recovery, "
        f"inj_id = {INJ_ID}\n"
        f"{plot_case['posterior_source']} posterior; "
        f"reference = {CENTER_MODE}"
    )

    fig, ax = make_violin_figure(
        normalized_plot=plot_case["normalized_plot"],
        normalized_full=plot_case["normalized_full"],
        cv_points=plot_case["cv_points"],
        pe_medians=plot_case["pe_medians"],
        labels=plot_case["labels"],
        title=title,
    )

    png_path = (
        PER_RUN_DIR
        / (
            f"{active_fti}_posterior_violin_"
            f"cv_bias_inj_id_{INJ_ID}.png"
        )
    )
    pdf_path = png_path.with_suffix(".pdf")

    if save:
        fig.savefig(
            png_path,
            dpi=SAVE_DPI,
            bbox_inches="tight",
        )
        fig.savefig(
            pdf_path,
            bbox_inches="tight",
        )

    if show:
        plt.show()

    return fig, png_path, pdf_path

## 10. Create one violin plot per FTI recovery

In [12]:
created_rows = []

for plot_case in plot_cases:
    figure, png_path, pdf_path = plot_one_fti_run(
        plot_case,
        save=True,
        show=True,
    )
    plt.close(figure)

    created_rows.append({
        "active_fti_run": plot_case["active_fti"],
        "posterior_source": plot_case["posterior_source"],
        "number_of_parameters": len(
            plot_case["parameters"]
        ),
        "png_file": str(png_path),
        "pdf_file": str(pdf_path),
    })

created_table = pd.DataFrame(created_rows)

created_path = (
    TABLE_DIR
    / f"created_violin_plots_inj_id_{INJ_ID}.csv"
)
created_table.to_csv(created_path, index=False)

display(created_table)
print("Saved plot index:", created_path)

findfont: Font family 'Times New Roman' not found.
findfont: Font family ['Times New Roman'] not found. Falling back to DejaVu Sans.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family ['Times New Roman'] not found. Falling back to DejaVu Sans.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font famil

,active_fti_run,posterior_source,number_of_parameters,png_file,pdf_file
0,fti_dchi0v,processed,12,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
1,fti_dchi1v,processed,12,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
2,fti_dchi2v,processed,12,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
3,fti_dchi3v,processed,12,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
4,fti_dchi4v,processed,12,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
5,fti_dchi6lv,processed,12,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
6,fti_dchi6v,processed,12,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
7,fti_dchi7v,processed,12,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
8,fti_dchiMinus2v,processed,12,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
9,fti_dkappa_S,processed,12,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...


Saved plot index: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/tables/created_violin_plots_inj_id_6.csv


## 11. Save all recovery panels in one PDF

In [13]:
all_runs_pdf = (
    OUTPUT_ROOT
    / (
        f"all_fti_posterior_violin_"
        f"cv_bias_inj_id_{INJ_ID}.pdf"
    )
)

with PdfPages(all_runs_pdf) as pdf:
    for plot_case in plot_cases:
        figure, _, _ = plot_one_fti_run(
            plot_case,
            save=False,
            show=False,
        )

        pdf.savefig(
            figure,
            bbox_inches="tight",
        )
        plt.close(figure)

print("Saved multipage PDF:", all_runs_pdf)

findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times Ne

Saved multipage PDF: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/all_fti_posterior_violin_cv_bias_inj_id_6.pdf


## 12. Active-FTI-only summary plot

In [14]:
active_entries = []

for plot_case in plot_cases:
    active_fti = plot_case["active_fti"]

    if active_fti not in plot_case["parameters"]:
        continue

    index = plot_case["parameters"].index(active_fti)

    active_entries.append({
        "active_fti": active_fti,
        "label": parameter_label(active_fti),
        "normalized_full": (
            plot_case["normalized_full"][index]
        ),
        "normalized_plot": (
            plot_case["normalized_plot"][index]
        ),
        "cv_point": plot_case["cv_points"][index],
        "pe_median": plot_case["pe_medians"][index],
    })

if not active_entries:
    raise ValueError(
        "No active-FTI posterior entries were available."
    )

active_normalized_plot = [
    entry["normalized_plot"]
    for entry in active_entries
]
active_normalized_full = [
    entry["normalized_full"]
    for entry in active_entries
]
active_cv_points = np.asarray([
    entry["cv_point"]
    for entry in active_entries
])
active_pe_medians = np.asarray([
    entry["pe_median"]
    for entry in active_entries
])
active_labels = [
    entry["label"]
    for entry in active_entries
]

figure, axis = make_violin_figure(
    normalized_plot=active_normalized_plot,
    normalized_full=active_normalized_full,
    cv_points=active_cv_points,
    pe_medians=active_pe_medians,
    labels=active_labels,
    title=(
        f"Active FTI posterior distributions and "
        f"CV biases, inj_id = {INJ_ID}"
    ),
)

active_png = (
    OUTPUT_ROOT
    / (
        f"active_fti_posterior_violin_"
        f"cv_bias_inj_id_{INJ_ID}.png"
    )
)
active_pdf = active_png.with_suffix(".pdf")

figure.savefig(
    active_png,
    dpi=SAVE_DPI,
    bbox_inches="tight",
)
figure.savefig(
    active_pdf,
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

print("Saved:", active_png)
print("Saved:", active_pdf)

findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times Ne

Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/active_fti_posterior_violin_cv_bias_inj_id_6.png
Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/active_fti_posterior_violin_cv_bias_inj_id_6.pdf


## 13. Largest posterior-median versus CV differences

In [15]:
largest_differences = (
    summary_table.assign(
        absolute_PE_median_minus_CV=lambda frame: (
            frame["PE_median_minus_CV"].abs()
        )
    )
    .sort_values(
        "absolute_PE_median_minus_CV",
        ascending=False,
    )
    .reset_index(drop=True)
)

largest_path = (
    TABLE_DIR
    / (
        f"largest_pe_median_minus_cv_"
        f"inj_id_{INJ_ID}.csv"
    )
)
largest_differences.to_csv(
    largest_path,
    index=False,
)

display(largest_differences.head(30))
print("Saved:", largest_path)

,inj_id,active_fti_run,parameter,posterior_source,reference_mode,reference_value,sigma_CV,CV_delta_theta,CV_bias_over_sigma,number_of_posterior_samples,PE_mean_over_sigma_CV,PE_median_over_sigma_CV,PE_q05_over_sigma_CV,PE_q16_over_sigma_CV,PE_q84_over_sigma_CV,PE_q95_over_sigma_CV,PE_median_minus_CV,raw_burnin_used,raw_burnin_reason,absolute_PE_median_minus_CV
0,6,fti_dchiMinus2v,Deltat,processed,truth,0.000000,0.089899,1.989243,22.127614,35584,13.590014,13.586899,11.926504,12.589285,14.597787,15.268225,-8.540715,0,processed posterior,8.540715
1,6,fti_dchi1v,Deltat,processed,truth,0.000000,0.092572,1.968633,21.266054,35584,13.211298,13.210227,11.467571,12.167638,14.260878,14.958939,-8.055827,0,processed posterior,8.055827
2,6,fti_dchi2v,Deltat,processed,truth,0.000000,0.094462,1.979739,20.958093,40000,13.179832,13.178937,11.395741,12.095842,14.253479,14.973337,-7.779156,0,processed posterior,7.779156
3,6,fti_dxi0v,Deltat,processed,truth,0.000000,0.095503,2.015488,21.103969,35584,13.564579,13.548706,11.846846,12.519043,14.617497,15.306584,-7.555263,0,processed posterior,7.555263
4,6,fti_dchi0v,Deltat,processed,truth,0.000000,0.095337,2.013462,21.119421,40000,13.596110,13.592157,11.873494,12.558889,14.643416,15.331864,-7.527264,0,processed posterior,7.527264
5,6,fti_dchi3v,Deltat,processed,truth,0.000000,0.097681,2.009268,20.569713,35584,13.321018,13.310253,11.474237,12.196924,14.452039,15.191907,-7.259460,0,processed posterior,7.259460
6,6,fti_dchi4v,Deltat,processed,truth,0.000000,0.101582,2.056458,20.244261,40000,13.495341,13.485147,11.655807,12.375566,14.598181,15.331479,-6.759114,0,processed posterior,6.759114
7,6,fti_dchi6lv,Deltat,processed,truth,0.000000,0.102705,2.075621,20.209490,40000,13.619137,13.617467,11.766918,12.491919,14.746301,15.474500,-6.592023,0,processed posterior,6.592023
8,6,fti_dkappa_S,Deltat,processed,truth,0.000000,0.105310,2.110281,20.038786,35584,13.725413,13.720159,11.892232,12.606366,14.838899,15.544324,-6.318627,0,processed posterior,6.318627
9,6,fti_dchi6v,Deltat,processed,truth,0.000000,0.108226,2.163223,19.988012,35584,13.960250,13.967207,12.168975,12.873871,15.031604,15.742884,-6.020805,0,processed posterior,6.020805


Saved: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/tables/largest_pe_median_minus_cv_inj_id_6.csv


## 14. List files created

In [16]:
created_files = sorted(
    path
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
)

print("This notebook writes only under:")
print(OUTPUT_ROOT)
print()
print("Number of files:", len(created_files))

for path in created_files:
    print(path)

This notebook writes only under:
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots

Number of files: 29
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/active_fti_posterior_violin_cv_bias_inj_id_6.pdf
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/active_fti_posterior_violin_cv_bias_inj_id_6.png
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/all_fti_posterior_violin_cv_bias_inj_id_6.pdf
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/per_fti_run/fti_dchi0v_posterior_violin_cv_bias_inj_id_6.pdf
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/per_fti_run/fti_dchi0v_posterior_violin_cv_bias_inj_id_6.png
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/cv_posterior_violin_plots/per_fti_run/fti_dchi1v_posterior_violin_cv_bias_in